In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import shapely

import pandas as pd
import geopandas as gpd
from pathlib import Path
import shapely

def process_parcels_safely():
    input_dir = Path('/Users/kuba/Desktop/grid research/parcels')
    output_dir = input_dir.parent / 'clean_parcels'
    output_dir.mkdir(exist_ok=True)

    target_cols = ['tax_amt', 'legal1', 'legal2', 'legal3', 'zoning', 'lan_val', 'imp_val', 'tot_val']

    # 1. Get all raw files
    all_raw_files = list(input_dir.glob('*.parquet'))

    # 2. Smart Resume: Filter out files we've already cleaned
    files_to_process = []
    for file_path in all_raw_files:
        clean_file_path = output_dir / f"clean_{file_path.name}"
        if not clean_file_path.exists():
            files_to_process.append(file_path)

    already_done = len(all_raw_files) - len(files_to_process)

    print(f"Total files found: {len(all_raw_files)}")
    print(f"Already processed: {already_done}")
    print(f"Remaining to process: {len(files_to_process)}\n")

    if not files_to_process:
        print("Everything is already clean! ✅")
        return

    # 3. Process only the remaining files
    for i, file_path in enumerate(files_to_process, 1):
        clean_file_path = output_dir / f"clean_{file_path.name}"
        print(f"[{i}/{len(files_to_process)}] Processing: {file_path.name}")

        try:
            df = pd.read_parquet(file_path)

            candidates = [c for c in df.columns if 'geo' in c.lower() or 'shape' in c.lower()]
            if not candidates:
                continue
            geo_col = candidates[0]

            cols_to_keep = [c for c in target_cols if c in df.columns] + [geo_col]
            df = df[cols_to_keep]

            legal_cols_present = [c for c in ['legal1', 'legal2', 'legal3'] if c in df.columns]
            if legal_cols_present:
                df['legal'] = df[legal_cols_present].apply(
                    lambda row: ' '.join(row.dropna().astype(str).str.strip()), axis=1
                )
                df = df.drop(columns=legal_cols_present)
                df['legal'] = df['legal'].replace('', None)

            sample_geom = df[geo_col].dropna().iloc[0] if not df[geo_col].dropna().empty else None
            if sample_geom is None:
                 continue

            if isinstance(sample_geom, bytes):
                geometry_series = gpd.GeoSeries.from_wkb(df[geo_col])
            elif isinstance(sample_geom, str) and sample_geom.strip().startswith('{'):
                valid_mask = df[geo_col].notna()
                geoms = shapely.from_geojson(df.loc[valid_mask, geo_col].values)
                geometry_series = pd.Series(index=df.index, dtype='object')
                geometry_series.loc[valid_mask] = geoms
                geometry_series = gpd.GeoSeries(geometry_series)
            elif isinstance(sample_geom, str):
                geometry_series = gpd.GeoSeries.from_wkt(df[geo_col])
            else:
                continue

            gdf = gpd.GeoDataFrame(df, geometry=geometry_series, crs="EPSG:4326")
            if geo_col != 'geometry':
                gdf = gdf.drop(columns=[geo_col])

            # SAVE IMMEDIATELY TO DISK
            gdf.to_parquet(clean_file_path)

        except Exception as e:
            print(f"   [!] Error processing {file_path.name}: {e}")

    print("\nAll files processed safely! ✅")

if __name__ == "__main__":
    process_parcels_safely()

Total files found: 2500
Already processed: 165
Remaining to process: 2335

[1/2335] Processing: sdp_data_exchange_SCM_glint_solar_spatial_record_domain_v1_2025_01_30_21_10_55_766000000001538.parquet
[2/2335] Processing: sdp_data_exchange_SCM_glint_solar_spatial_record_domain_v1_2025_01_30_21_10_55_766000000000032.parquet
[3/2335] Processing: sdp_data_exchange_SCM_glint_solar_spatial_record_domain_v1_2025_01_30_21_10_55_766000000001450.parquet
[4/2335] Processing: sdp_data_exchange_SCM_glint_solar_spatial_record_domain_v1_2025_01_30_21_10_55_766000000001528.parquet
[5/2335] Processing: sdp_data_exchange_SCM_glint_solar_spatial_record_domain_v1_2025_01_30_21_10_55_766000000000022.parquet
[6/2335] Processing: sdp_data_exchange_SCM_glint_solar_spatial_record_domain_v1_2025_01_30_21_10_55_766000000001215.parquet
[7/2335] Processing: sdp_data_exchange_SCM_glint_solar_spatial_record_domain_v1_2025_01_30_21_10_55_766000000000667.parquet
[8/2335] Processing: sdp_data_exchange_SCM_glint_solar_sp

In [5]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
from tqdm import tqdm

def process_and_split_by_state():
    # 1. Paths
    input_dir = Path('/Users/kuba/Desktop/grid research/clean_parcels')
    output_dir = Path('/Users/kuba/Desktop/grid research/state_parcels')
    output_dir.mkdir(parents=True, exist_ok=True)

    states_path = '/Users/kuba/PycharmProjects/grid-research/data/state_borders/cb_2023_us_state_20m.gpkg'

    # 2. Load State Borders
    print("Loading US State borders...")
    states = gpd.read_file(states_path, engine="pyogrio")

    # Project to EPSG:5070 (US National Equal Area, measured in meters)
    states = states.to_crs("EPSG:5070")

    # Explicitly use the STUSPS column
    state_col = 'STUSPS'
    if state_col not in states.columns:
        print(f"Error: Could not find '{state_col}'. Available columns: {list(states.columns)}")
        return

    states = states[[state_col, 'geometry']]

    # 3. Find input files
    parquet_files = list(input_dir.glob('*.parquet'))
    if not parquet_files:
        print("No clean parquet files found!")
        return

    print(f"Found {len(parquet_files)} clean parcel files. Beginning national simplification and partitioning...")

    # 4. Process loop with progress bar
    for file_path in tqdm(parquet_files, desc="Processing Parcels"):
        try:
            # Load chunk
            gdf = gpd.read_parquet(file_path)
            if gdf.empty:
                continue

            # STEP A: KEEP POLYGONS ONLY
            gdf = gdf[gdf.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])].copy()
            if gdf.empty:
                continue

            # STEP B: SIMPLIFY (5 METERS)
            # Ensure CRS is set before projecting, just in case
            if gdf.crs is None:
                gdf.set_crs("EPSG:4326", inplace=True)

            gdf = gdf.to_crs("EPSG:5070")
            gdf['geometry'] = gdf.geometry.simplify(5.0, preserve_topology=True)

            # STEP C: SPATIAL JOIN WITH STATES
            # We use 'intersects' so parcels touching a border are captured.
            joined = gpd.sjoin(gdf, states, how='inner', predicate='intersects')

            # If a parcel sits exactly on a border, it might duplicate. Keep only the first state it hit.
            joined = joined[~joined.index.duplicated(keep='first')]

            # Clean up the sjoin artifacts
            if 'index_right' in joined.columns:
                joined = joined.drop(columns=['index_right'])

            # STEP D: PROJECT BACK TO GPS
            joined = joined.to_crs("EPSG:4326")

            # STEP E: SAVE INTO STATE FOLDERS
            for state_abbr, state_gdf in joined.groupby(state_col):
                # Folder will be exactly the STUSPS code (e.g., /state_parcels/MA/)
                state_dir = output_dir / str(state_abbr)
                state_dir.mkdir(exist_ok=True)

                # Save this chunk of parcels into the state folder
                out_file = state_dir / file_path.name

                # Drop the state column to save file size (since it's in the folder name now)
                state_gdf = state_gdf.drop(columns=[state_col])
                state_gdf.to_parquet(out_file)

        except Exception as e:
            print(f"\n[!] Error processing {file_path.name}: {e}")

    print("\nAll files successfully simplified, cleaned, and partitioned by state! ✅")

if __name__ == "__main__":
    process_and_split_by_state()

Loading US State borders...
Found 2500 clean parcel files. Beginning national simplification and partitioning...


Processing Parcels: 100%|██████████| 2500/2500 [58:17<00:00,  1.40s/it] 


All files successfully simplified, cleaned, and partitioned by state! ✅


In [7]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import zipfile
from tqdm import tqdm

def merge_and_zip_target_states():
    # 1. Define your target states (New England + PA + DE)
    target_states = ['ME', 'NH', 'VT', 'MA', 'RI', 'CT', 'PA', 'DE']

    # 2. Paths based on your previous output
    input_base_dir = Path('/Users/kuba/Desktop/grid research/state_parcels')

    # Create a new directory for the final merged & zipped files
    output_dir = Path('/Users/kuba/Desktop/grid research/final_merged_states')
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Starting merge process for: {', '.join(target_states)}\n")

    # 3. Process loop for each state
    for state in target_states:
        state_dir = input_base_dir / state

        # Check if the folder actually exists and has files
        if not state_dir.exists():
            print(f"[!] Skipping {state}: Directory not found at {state_dir}")
            continue

        parquet_files = list(state_dir.glob('*.parquet'))
        if not parquet_files:
            print(f"[!] Skipping {state}: No Parquet files found in directory.")
            continue

        print(f"--- Processing {state} ({len(parquet_files)} files) ---")

        # Step A: Load all chunks for this state
        gdfs = []
        for file in tqdm(parquet_files, desc=f"Loading chunks for {state}"):
            try:
                gdf = gpd.read_parquet(file)
                if not gdf.empty:
                    gdfs.append(gdf)
            except Exception as e:
                print(f"\n[!] Error reading {file.name}: {e}")

        if not gdfs:
            print(f"[!] No valid data found for {state} after reading. Skipping.")
            continue

        # Step B: Concatenate into one master GeoDataFrame
        print(f"Concatenating {state} data...")
        merged_gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))

        # Set the CRS just to be absolutely safe, based on your previous script's output
        if merged_gdf.crs is None:
            merged_gdf.set_crs("EPSG:4326", inplace=True)

        # Define output paths
        merged_parquet_path = output_dir / f"{state}_master_parcels.parquet"
        zipped_path = output_dir / f"{state}_master_parcels.zip"

        # Step C: Save to a single Parquet file
        print(f"Saving {state} master Parquet...")
        merged_gdf.to_parquet(merged_parquet_path)

        # Step D: Zip the Parquet file
        print(f"Zipping {state} master file...")
        with zipfile.ZipFile(zipped_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            # arcname ensures the zip just contains the file, not the whole folder path structure
            zipf.write(merged_parquet_path, arcname=merged_parquet_path.name)

        # Optional: Delete the unzipped master parquet file if you only want to keep the ZIP
        # merged_parquet_path.unlink()

        print(f"✅ {state} Complete! Saved to: {zipped_path}\n")

    print("🎉 All target states have been merged and zipped!")

if __name__ == "__main__":
    merge_and_zip_target_states()

Starting merge process for: ME, NH, VT, MA, RI, CT, PA, DE

--- Processing ME (2500 files) ---


Loading chunks for ME: 100%|██████████| 2500/2500 [00:05<00:00, 425.89it/s]


Concatenating ME data...
Saving ME master Parquet...
Zipping ME master file...
✅ ME Complete! Saved to: /Users/kuba/Desktop/grid research/final_merged_states/ME_master_parcels.zip

--- Processing NH (2500 files) ---


Loading chunks for NH: 100%|██████████| 2500/2500 [00:09<00:00, 271.85it/s]


Concatenating NH data...
Saving NH master Parquet...
Zipping NH master file...
✅ NH Complete! Saved to: /Users/kuba/Desktop/grid research/final_merged_states/NH_master_parcels.zip

--- Processing VT (2500 files) ---


...

Concatenating CT data...
Saving CT master Parquet...
Zipping CT master file...
✅ CT Complete! Saved to: /Users/kuba/Desktop/grid research/final_merged_states/CT_master_parcels.zip

--- Processing PA (2500 files) ---


Loading chunks for PA: 100%|██████████| 2500/2500 [00:10<00:00, 231.60it/s]


Concatenating PA data...
Saving PA master Parquet...
Zipping PA master file...
✅ PA Complete! Saved to: /Users/kuba/Desktop/grid research/final_merged_states/PA_master_parcels.zip

--- Processing DE (2500 files) ---


Loading chunks for DE: 100%|██████████| 2500/2500 [00:05<00:00, 452.25it/s]


Concatenating DE data...
Saving DE master Parquet...
Zipping DE master file...
✅ DE Complete! Saved to: /Users/kuba/Desktop/grid research/final_merged_states/DE_master_parcels.zip

🎉 All target states have been merged and zipped!
